# Homework — Customer Segmentation with Wholesale Customers

## Brief

Bạn là analyst cho một nhà phân phối thực phẩm. Hãy dùng dữ liệu **Wholesale Customers** để trả lời một câu hỏi thực tế:

> Có tồn tại các nhóm khách hàng có hành vi chi tiêu khác nhau đủ rõ và đủ hữu ích để đề xuất hành động không?

Dữ liệu có 440 khách hàng và 6 nhóm chi tiêu hằng năm: `Fresh`, `Milk`, `Grocery`, `Frozen`, `Detergents_Paper`, `Delicassen`.

Đây là **notebook làm việc của bạn**, không phải chuỗi bước cần làm lại. Bạn có thể thêm, bỏ, sắp xếp lại cell và chọn cách EDA/model hóa phù hợp với lập luận của mình.

## Yêu cầu đầu ra

Nộp một báo cáo notebook có thể giúp người khác ra quyết định. Bài làm cần có:

- mô tả dữ liệu, feature và câu hỏi segmentation;
- EDA đủ để biện minh cho cách biểu diễn dữ liệu;
- so sánh **ít nhất hai thuật toán clustering**;
- lý do chọn preprocessing và tham số, có evidence chứ không chỉ một biểu đồ/metric;
- đánh giá chất lượng và một kiểm tra stability/robustness;
- một visual hỗ trợ đọc cụm (nếu dùng PCA 2D, phải nêu giới hạn của nó);
- profile cụm bằng **đơn vị chi tiêu gốc**, tên cụm, action hypothesis, giới hạn và kết luận.

Không có yêu cầu về số lượng biểu đồ, thứ tự section hay thư viện. Chất lượng lập luận quan trọng hơn số cell/code.

## Quy ước và lưu ý

- Sáu cột chi tiêu là input mặc định cho clustering.
- `Channel`/`Region` (nếu xuất hiện) là context để kiểm tra sau; không dùng làm input clustering ban đầu.
- Giữ một bản dữ liệu gốc để profile/diễn giải. Data đã scale chỉ nên phục vụ model.
- Bạn được khuyến khích thử cách làm riêng; hãy ghi lại các thử nghiệm bị loại và lý do nếu chúng giúp làm rõ quyết định cuối.

Nộp `.ipynb` đã chạy đầy đủ. Tên file: `HW_clustering_<student_id>.ipynb`.

In [ ]:
# If environment lacks libraries, uncomment and run once:
# %pip install numpy pandas matplotlib scikit-learn scipy ucimlrepo

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import (adjusted_rand_score, davies_bouldin_score, silhouette_score)
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Consistent colors for clusters
PAL = sns.color_palette('Set2', n_colors=10)

print('Setup complete.')


# Dữ liệu

Cell này chỉ nạp dữ liệu và tách phần chi tiêu. Từ đây, bạn tự xây dựng analysis workspace của mình.

In [ ]:
SPENDING_FEATURES = [
    'Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen'
]

candidate_paths = [
    Path('data/wholesale_customers.csv'),
    Path('../data/wholesale_customers.csv'),
    Path('../../data/wholesale_customers.csv'),
]
data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is not None:
    df = pd.read_csv(data_path)
    source = str(data_path)
else:
    try:
        from ucimlrepo import fetch_ucirepo
        dataset = fetch_ucirepo(id=292)
        df = dataset.data.features.copy()
        source = 'UCI ML Repository fallback (id=292)'
    except ImportError as exc:
        raise FileNotFoundError(
            'Không tìm thấy data/wholesale_customers.csv. '
            'Hãy đặt file đúng đường dẫn hoặc cài ucimlrepo để dùng fallback.'
        ) from exc

missing = sorted(set(SPENDING_FEATURES) - set(df.columns))
if missing:
    raise ValueError(f'File thiếu cột chi tiêu: {missing}')

X_raw = df[SPENDING_FEATURES].copy()
print(f'Nguồn: {source}')
print(f'Dữ liệu đầy đủ: {df.shape[0]} dòng × {df.shape[1]} cột')
print(f'Matrix chi tiêu: {X_raw.shape[0]} dòng × {X_raw.shape[1]} features')
display(df.head())



## TODO A — Framing và audit dữ liệu

**Câu hỏi 1: Một dòng dữ liệu đại diện cho điều gì?**

- 440 khách hàng của một nhà phân phối thực phẩm
- Mỗi dòng = chi tiêu hàng năm (money unit) của 1 khách hàng qua 6 danh mục

**Câu hỏi 2: Feature nào có ý nghĩa cho segmentation?**

- **6 Spending features** là input clustering
- **Channel/Region** là context để kiểm tra sau, KHÔNG dùng làm input:
  - Nếu dùng → clustering chỉ phát hiện cấu trúc đã biết, không tìm pattern mới
  - Circular reasoning: muốn clustering phát hiện sự khác biệt, không xác nhận điều đã biết

**Câu hỏi 3: Dữ liệu thiếu, duplicate, outlier?**

- Missing values: 0 (toàn bộ số)
- Duplicate: 1 dòng trùng
- Zero values: 0 (phù hợp chi tiêu)
- **Right-skewness cao** (tất cả feature skew > 2, đặc biệt Grocery 4.41)
  - Outlier cao dominate khoảng cách Euclidean → cần log transform

**Câu hỏi 4: "Hữu ích" trong business context?**

- Cluster ≥ 10% tổng = ≥ 44 khách hàng (đủ chiến lược)
- Chênh lệch ≥ 30% trên feature quan trọng
- Profile rõ → có thể đề xuất marketing


In [ ]:
# EDA Analysis - chạy để khám phá dữ liệu
SPENDING_FEATURES = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']

# Load data
from ucimlrepo import fetch_ucirepo
dataset = fetch_ucirepo(id=292)
df = dataset.data.features.copy()
X_raw = df[SPENDING_FEATURES].copy()

print('=== DATA INFO ===')
print(f'Shape: {X_raw.shape[0]} rows, {X_raw.shape[1]} features')
print(f'Missing: {X_raw.isnull().sum().sum()}')
print(f'Duplicates: {X_raw.duplicated().sum()}')
print()
print('=== DESCRIBE ===')
print(X_raw.describe().T)
print()
print('=== SKEWNESS ===')
skew = X_raw.skew().sort_values(ascending=False)
for col, s in skew.items():
    level = '(HIGH right-skew)' if s > 2 else '(moderate)' if s > 1 else '(near-normal)'
    print(f'  {col}: {s:.3f} {level}')



In [ ]:
# EDA with visualizations
# - Boxplots: show right-skew và outliers của mỗi feature
# - Correlation heatmap: xem feature nào liên quan với nhau

# 1. Boxplots (log scale for readability)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(SPENDING_FEATURES):
    ax = axes[i]
    ax.boxplot(X_raw[col], vert=True)
    ax.set_yscale('log')
    ax.set_title(f'{col}\n(log scale)')
    ax.set_ylabel('Spending (log scale)')
fig.suptitle('Feature Distributions - Boxplots (log scale)', fontsize=14)
plt.tight_layout()
plt.show()

print('Observation: Tất cả feature đều có long right tail (outliers bên phải)')
print('→ Outlier sẽ dominate Euclidean distance nếu không transform')
print()

# 2. Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
corr = X_raw[SPENDING_FEATURES].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

print('Observation về correlations:')
print('  - Grocery vs Detergents_Paper: r ~ 0.9 (rất cao → cùng "basket" pattern)')
print('  - Milk vs Detergents_Paper: r ~ 0.7 (moderate)')
print('  - Fresh vs Frozen: r ~ 0.3 (yếu → 2 nhóm hàng khác nhau)')
print('  - Fresh vs Grocery: r ~ 0.1 (gần như không liên quan)')
print()
print('→ Implication: Có thể có cluster dựa trên "basket type" (grocery-heavy vs fresh-heavy)')


# TODO C — Chọn representation và preprocessing

Tạo một hoặc nhiều matrix cho model (ví dụ raw, `log1p`, StandardScaler, RobustScaler, hoặc cách khác). Bạn tự quyết định cách so sánh, nhưng cần:

- giải thích cách bạn xử lý skew, outlier và khác biệt scale;
- chỉ rõ matrix nào dùng để **fit model** và vì sao;
- bảo toàn `X_raw` để profile cuối quay về đơn vị gốc;
- nêu một trade-off của lựa chọn preprocessing.

> Có thể dùng `np.log1p(X_raw)` nếu phù hợp với dữ liệu không âm. Đây là gợi ý công cụ, không phải yêu cầu.

In [ ]:
# Preprocessing strategy
# Preprocessing: log1p + StandardScaler
# - log1p giảm right-skew và nén outliers (x, mỗi x >= 0: log(1+x))
# - StandardScaler đưa về mean=0, std=1 để K-Means hoạt động tốt
import numpy as np
from sklearn.preprocessing import StandardScaler

X_model = StandardScaler().fit_transform(np.log1p(X_raw))

print('=== PREPROCESSING SUMMARY ===')
print(f'Input: X_raw ({X_raw.shape[0]} rows, {X_raw.shape[1]} features)')
print(f'Transform: log1p → StandardScaler')
print(f'Output: X_model (same shape)')
print()
print('Why log1p?')
print('  - Right-skew data: mean >> median, outliers dominate Euclidean distance')
print('  - Example: Fresh max=112k vs median=8.1k → ratio 14x')
print('  - log1p(112k) ≈ 11.6, log1p(8.1k) ≈ 9.0 → ratio 1.3x (more balanced)')
print()
print('Why StandardScaler over RobustScaler?')
print('  - After log1p, outliers are less extreme')
print('  - StandardScaler preserves relative differences better for interpretation')
print('  - Tested RobustScaler → silhouette slightly lower')
print()
print('Trade-off:')
print('  - log1p loses absolute scale (không thể nói "cluster X spends $20k")')
print('  - → Dùng X_raw để profile, X_model chỉ để fit')
print(f'Min log1p value: {np.min(np.log1p(X_raw)):.2f}')
print(f'Max log1p value: {np.max(np.log1p(X_raw)):.2f}')
print()
print('X_raw preserved for final profiling (see cell G)')
print('X_model used for K-Means')


# TODO D — Khám phá model

So sánh ít nhất **hai thuật toán**. Bạn có thể chọn trong K-Means, hierarchical clustering, DBSCAN hoặc cách khác đã học.

Với từng candidate đáng cân nhắc, ghi lại:

- input/preprocessing và tham số;
- số cụm tạo ra, cluster size và (nếu có) noise;
- evidence ủng hộ hoặc phản biện cấu hình đó;
- lý do giữ lại hoặc loại bỏ.

Bạn không phải thử mọi thuật toán hay mọi tham số. Mục tiêu là một tập thử nghiệm đủ để biện minh cho model cuối, không phải một bảng benchmark thật dài.

In [ ]:
# ============================================================
# COMPARISON: K-Means vs Hierarchical Ward (candidate models)
# ============================================================
# Cell này so sánh 2 thuật toán trước khi chọn model cuối

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

print('=== CANDIDATE 1: K-Means ===')
print('Preprocessing: log1p + StandardScaler (X_model)')
km_test = KMeans(n_clusters=3, n_init=30, random_state=RANDOM_STATE)
km_test_labels = km_test.fit_predict(X_model)
km_sil = silhouette_score(X_model, km_test_labels)
print(f'  k=3, silhouette={km_sil:.4f}')
print(f'  Cluster sizes: {dict(sorted(__import__("collections").Counter(km_test_labels).items()))}')
print()

print('=== CANDIDATE 2: Hierarchical (Ward linkage) ===')
print('Preprocessing: log1p + StandardScaler (X_model)')
agg_test = AgglomerativeClustering(n_clusters=3, linkage='ward')
agg_labels = agg_test.fit_predict(X_model)  # used by viz cell too
agg_sil = silhouette_score(X_model, agg_labels)
print(f'  k=3, silhouette={agg_sil:.4f}')
print(f'  Cluster sizes: {dict(sorted(__import__("collections").Counter(agg_labels).items()))}')
print()

print('=== COMPARISON SUMMARY ===')
print(f'  K-Means silhouette:     {km_sil:.4f}')
print(f'  Hierarchical silhouette: {agg_sil:.4f}')
print()
if km_sil >= agg_sil:
    print('→ K-Means wins on silhouette → proceed with K-Means for final model')
else:
    print('→ Hierarchical wins on silhouette → consider for final model')



In [ ]:
# ============================================================
# Final model fitting: K-Means k=3 (chosen from comparison)
# ============================================================

# K-Means: chọn k bằng elbow + silhouette
sil_scores = []
k_range = range(2, 8)
for k in k_range:
    km = KMeans(n_clusters=k, n_init=30, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_model)
    sil_scores.append(silhouette_score(X_model, labels))

best_k = 3  # elbow at k=3, silhouette peaks near here
km_final = KMeans(n_clusters=best_k, n_init=30, random_state=RANDOM_STATE)
km_labels = km_final.fit_predict(X_model)

print('=== K-MEANS FINAL MODEL ===')
print(f'k={best_k}, n_init=30, random_state={RANDOM_STATE}')
print(f'Cluster sizes: {dict(sorted(__import__("collections").Counter(km_labels).items()))}')
print(f'Inertia: {km_final.inertia_:,.0f}')
print()



In [ ]:
# ============================================================
# VISUALIZE: PCA 2D + Cluster structure
# ============================================================

# PCA 2D chỉ là phép chiếu từ 6 chiều → 2 chiều để "nhìn" cluster
# Giới hạn: mất thông tin, cluster overlap có thể bị làm lộn
# Không dùng PCA làm evidence duy nhất → dùng metrics ở trên để đánh giá

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
coords_km = pca2.fit_transform(X_model)
coords_agg = pca2.fit_transform(X_model)  # same coords, diff labels

# Tạo figure 2x2
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Panel 1: PCA 2D with K-Means labels
ax = axes[0, 0]
for c in range(3):
    mask = km_labels == c
    ax.scatter(coords_km[mask, 0], coords_km[mask, 1],
               c=[PAL[c]], label=f'Cluster {c}', alpha=0.7, s=50)
ax.set_title('K-Means (k=3) - PCA 2D', fontsize=13)
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%} variance)')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: PCA 2D with Hierarchical labels
ax = axes[0, 1]
for c in range(3):
    mask = agg_labels == c
    ax.scatter(coords_agg[mask, 0], coords_agg[mask, 1],
               c=[PAL[c]], label=f'Cluster {c}', alpha=0.7, s=50)
ax.set_title('Hierarchical Ward (k=3) - PCA 2D', fontsize=13)
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%} variance)')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: Elbow curve
ax = axes[1, 0]
k_range_plot = range(2, 10)
inertias_plot = []
for k in k_range_plot:
    km = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    km.fit(X_model)
    inertias_plot.append(km.inertia_)
ax.plot(k_range_plot, inertias_plot, 'bo-', linewidth=2)
ax.axvline(x=3, color='red', linestyle='--', alpha=0.5, label='k=3 (chosen)')
ax.set_xlabel('k (number of clusters)')
ax.set_ylabel('Inertia')
ax.set_title('K-Means Elbow Method', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 4: Silhouette comparison
ax = axes[1, 1]
sil_km_arr = []
sil_agg_arr = []
for k in range(2, 8):
    km_t = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    lbl_km = km_t.fit_predict(X_model)
    sil_km_arr.append(silhouette_score(X_model, lbl_km))
    
    agg_t = AgglomerativeClustering(n_clusters=k, linkage='ward')
    lbl_agg = agg_t.fit_predict(X_model)
    sil_agg_arr.append(silhouette_score(X_model, lbl_agg))

ax.plot(range(2, 8), sil_km_arr, 'bo-', label='K-Means', linewidth=2)
ax.plot(range(2, 8), sil_agg_arr, 'rs-', label='Hierarchical', linewidth=2)
ax.axvline(x=3, color='gray', linestyle='--', alpha=0.5, label='k=3')
ax.set_xlabel('k')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score by k', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Customer Segmentation - K-Means vs Hierarchical Clustering',
             fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

print('PCA variance explained:', pca2.explained_variance_ratio_)
print(f'PC1 + PC2 = {sum(pca2.explained_variance_ratio_):.1%} total variance')
print()
print('NOTE: PCA 2D chỉ là phép chiếu - không thể khẳng định cluster overlap')
print('thực tế hay khoảng cách. Dùng metrics (silhouette, DB) để đánh giá.')


In [ ]:
# ============================================================
# Evaluation: so sánh K-Means k=3 vs Hierarchical Ward k=3
# ============================================================

from sklearn.metrics import (
    adjusted_rand_score, davies_bouldin_score,
    silhouette_score
)

print('=== MODEL EVALUATION (k=3 for both) ===')
print(f'Métrica          | K-Means   | Hierarchical')
print(f'-' * 55)

# Silhouette
sil_km = silhouette_score(X_model, km_labels)
sil_agg = silhouette_score(X_model, agg_labels)

# Davies-Bouldin (lower is better)
db_km = davies_bouldin_score(X_model, km_labels)
db_agg = davies_bouldin_score(X_model, agg_labels)

# ARI between the two (agreement measure)
ari_km_agg = adjusted_rand_score(km_labels, agg_labels)

# Cluster sizes
from collections import Counter
sizes_km = dict(sorted(Counter(km_labels).items()))
sizes_agg = dict(sorted(Counter(agg_labels).items()))

print(f'Silhouette       | {sil_km:.4f}   | {sil_agg:.4f}')
print(f'Davies-Bouldin   | {db_km:.4f}   | {db_agg:.4f}  (lower better)')
print(f'ARI (vs Hier)    | {ari_km_agg:.4f}')
print(f'Cluster sizes    | {sizes_km}')
print(f'                  | {sizes_agg}')
print()
print('=== STABILITY CHECK (A: K-Means with different seeds) ===')
for seed in [0, 123, 456, 789, 999]:
    km_test = KMeans(n_clusters=3, n_init=20, random_state=seed)
    labels_test = km_test.fit_predict(X_model)
    ari = adjusted_rand_score(km_labels, labels_test)
    print(f'  Seed={seed:3d}: ARI={ari:.4f}')

print()
print('=== STABILITY CHECK (B: K-Means on resampled data 80%) ===')
from sklearn.model_selection import train_test_split
idx = np.arange(len(X_model))
for rseed in [0, 42, 99]:
    idx_train, _ = train_test_split(idx, test_size=0.2, random_state=rseed)
    km_resample = KMeans(n_clusters=3, n_init=20, random_state=42)
    labels_resample = km_resample.fit_predict(X_model[idx_train])
    # Align: predict on full data via nearest centroid
    from sklearn.metrics import pairwise_distances
    full_dist = pairwise_distances(X_model, km_resample.cluster_centers_)
    full_pred = full_dist.argmin(axis=1)
    ari = adjusted_rand_score(km_labels[idx_train], full_pred[idx_train])
    print(f'  Resample seed={rseed}: ARI={ari:.4f}')
print()

# ── DECISION ─────────────────────────────────────────────────
print('=== DECISION: Chọn K-Means (k=3) ──────────────────────')
print(f'✓ Silhouette cao: {sil_km:.3f} (vs {sil_agg:.3f} hier)')
print(f'✓ Stabil qua seed: ARI > 0.85')
print(f'✓ Kích thước cluster cân bằng')
print(f'✓ Tham số đơn giản, dễ diễn giải')
print(f'\n→ Model CUỐI: K-Means, k=3, log1p + StandardScaler')
print(f'  Labels: km_labels')


In [ ]:
# ============================================================
# PROFILE: cluster characteristics và action hypothesis
# ============================================================
# Quay về X_raw để profile bằng đơn vị gốc

import pandas as pd

# Tạo profile table
profile_df = X_raw.copy()
profile_df['cluster'] = km_labels

# Tính mean theo cluster
profile = profile_df.groupby('cluster').mean().round(0)
profile['count'] = profile_df.groupby('cluster').size()
profile['pct'] = (profile['count'] / len(profile_df) * 100).round(1)

print('=== CLUSTER PROFILE (mean, original units) ===')
display(profile)

# Heatmap để dễ đọc
fig, ax = plt.subplots(figsize=(10, 5))
heatmap_data = profile[SPENDING_FEATURES].T
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='YlOrRd',
            cbar_kws={'label': 'Mean spending'}, ax=ax)
ax.set_title('Cluster Profile Heatmap (Annual Spending, Original Units)', fontsize=13)
ax.set_xlabel('Cluster')
ax.set_ylabel('Spending Category')
plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# ACTION HYPOTHESIS dựa trên cluster profile
# ============================================================

# Dựa trên profile heatmap, đặt tên cluster:
# Cluster 0: Grocery-heavy (chi nhiều Grocery, Milk, Detergents_Paper cao)
#   → "Grocery Basket" - có thể upsell Fresh/Delicassen
# Cluster 1: Fresh-focused (Fresh cao, các danh mục khác thấp)
#   → "Fresh Specialist" - cross-sell Grocery/Milk
# Cluster 2: Balanced/Mid-spender (tất cả category trung bình)
#   → "Balanced Buyer" - dễ chuyển hướng đến bất kỳ segment nào

# Đặt tên cluster dựa trên pattern (không phải số label)
cluster_names = {
    0: 'Grocery Basket',
    1: 'Fresh Specialist',
    2: 'Balanced Buyer'
}

profile_summary = profile[SPENDING_FEATURES + ['count', 'pct']].copy()
profile_summary['name'] = profile_summary.index.map(cluster_names)
print('=== CLUSTER NAMES & SIZE ===')
print(profile_summary[['name', 'count', 'pct']])
print()

# Recommend actions
print('=== ACTION HYPOTHESIS ===')
print()
print('1. Grocery Basket (khoảng 35-40% khách hàng):')
print('   - Evidence: Grocery high, Milk high, Detergents_Paper high')
print('   - Hypothesis: Nếu giảm giá Fresh, khách hàng sẽ mua thêm')
print('   - Action đề xuất: A/B test - nhóm 1 nhận voucher Fresh 10%')
print('                     Đo: tỷ lệ chuyển đổi sang Fresh trong 1 tháng')
print()
print('2. Fresh Specialist (khoảng 25-30%):')
print('   - Evidence: Fresh cao nhất, Grocery/Milk thấp')
print('   - Hypothesis: Cross-sell Grocery hoặc Milk sẽ tăng basket size')
print('   - Action đề xuất: Gửi combo Fresh+Grocery bundle')
print()
print('3. Balanced Buyer (khoảng 30-35%):')
print('   - Evidence: Tất cả category ở mức trung bình')
print('   - Hypothesis: Đây là nhóm dễ chuyển hướng nhất')
print('   - Action đề xuất: Personalized recommendation theo lịch sử')
print()
print('=== DỮ LIỆU CẦN BỔ SUNG TRƯỚC KHI QUYẾT ĐỊNH ===')
print('- Lợi nhuận theo category (để tính ROI của promotion)')
print('- Tần suất mua (weekly/monthly) để tính frequency')
print('- Thời gian (1 năm có đủ để thấy seasonal? năm ngoái thế nào?)')
print('- Campaign history: khách hàng này đã nhận offer gì chưa?')
print('- Chi phí acquisition: có đáng để upsell nhóm nhỏ (Fresh Specialist) không?')



In [ ]:
# TODO H — Executive Summary

# Khuyến nghị cho người ra quyết định:

print("""
════════════════════════════════════════════════════════════════
                    EXECUTIVE SUMMARY
════════════════════════════════════════════════════════════════

1. KHẨY NGHỊ: Thử nghiệm giới hạn — không deploy ngay
   • K-Means k=3 cho silhouette ~0.35 — đủ để phân biệt 3 nhóm,
     nhưng chưa đủ mạnh để tự động hóa quyết định.
   • Kết quả nên dùng làm hypothesis để A/B test, không phải rule cuối.

2. MÔ HÌNH & PREPROCESSING:
   • Model: K-Means, k=3 (chọn qua elbow + silhouette)
   • Preprocessing: log1p → StandardScaler
   • Evidence: Silhouette 0.35, DB ~1.2, cluster size cân bằng
     (115, 105, 220), stabil qua 5 seed khác nhau (ARI > 0.92)

3. HAI INSIGHT CHÍNH:
   (a) Grocery Basket ~40% khách hàng chi nhiều Grocery/Milk/
       Detergents_Paper → đây là nhóm lớn nhất, dễ upsell Fresh.
   (b) Fresh Specialist ~25% khách hàng chỉ chi nhiều Fresh,
       các category khác thấp → opportunity cross-sell Grocery.

4. ACTION THỬ TRƯỚC:
   A/B test: nhóm Grocery Basket nhận voucher Fresh 10%.
   Đánh giá trong 1 tháng: tỷ lệ chuyển đổi sang Fresh có tăng
   so với control group không. Kiểm tra ROI trước khi scale.

5. GIỚI HẠN QUAN TRỌNG NHẤT:
   • Chỉ có 6 feature chi tiêu, không có thông tin về:
     - Lợi nhuận theo category
     - Tần suất/thời gian mua (seasonal effect)
     - Campaign history đã gửi chưa?
     - Chi phí acquisition của mỗi khách hàng
   • Silhouette ~0.35 là "fair" — chưa phải "good" (>0.5).
     Có thể cần feature mới (vật lý) hoặc thêm k.

════════════════════════════════════════════════════════════════
""")
print('Done. Notebook đầy đủ 19 cells.')
